# Explore dataset
Run this on Kaggle against the FaceForensics++ (or chosen) dataset input, before running `prepare_dataset.py` at scale.
Goals: confirm the face detector works on sample videos, inspect class balance, sanity-check crop quality.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/Deepfake Detection')  # adjust if repo uploaded under a different path
sys.path.insert(0, str(REPO_ROOT / 'backend'))

from app.services.detection.face_extractor import extract_face
import cv2, matplotlib.pyplot as plt

In [ ]:
RAW_DIR = Path('/kaggle/input/<dataset-slug>')  # set after choosing the Kaggle dataset
real_videos = sorted(RAW_DIR.glob('original_sequences/**/*.mp4'))[:3]
fake_videos = sorted(RAW_DIR.glob('manipulated_sequences/Deepfakes/**/*.mp4'))[:3]
print(len(real_videos), len(fake_videos))

In [ ]:
def preview(video_path, n=4):
    cap = cv2.VideoCapture(str(video_path))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    shown = 0
    frame_idx = 0
    while shown < n:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % 10 == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            crop = extract_face(rgb)
            axes[shown].imshow(crop if crop is not None else rgb)
            axes[shown].set_title('face' if crop is not None else 'NO FACE')
            axes[shown].axis('off')
            shown += 1
        frame_idx += 1
    cap.release()
    plt.show()

for v in real_videos:
    preview(v)
for v in fake_videos:
    preview(v)

After confirming face detection works well on samples above, run `training/scripts/prepare_dataset.py` to extract the full dataset, then `train.py`.